# 01 - Fetch HotCRP Members

In this notebook, I fetch the public HotCRP `/users/pc` pages for ICFP,
POPL, OOPSLA, and PLDI.

I use this data to understand who was listed in the review system. Later, I
will compare it with the visible conference website lists.


## 1 - Setup

In [1]:
import re
import requests

import pandas as pd

from bs4 import BeautifulSoup
from datetime import date
from pathlib import Path


In [2]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

from project_setup import setup_project

setup = setup_project()
project_folder = setup.project_folder
PROJECT = project_folder
repo = project_folder
config_path = setup.config_path
project_config = setup.project_config

run_mode = setup.run_mode
inputs_config = setup.inputs
outputs_config = setup.outputs
openalex_config = setup.openalex

allow_network = setup.allow_network
use_existing_data = setup.use_existing_data
overwrite_data = setup.overwrite_data
overwrite_artifacts = setup.overwrite_artifacts
openalex_sample_limit = setup.openalex_sample_limit
openalex_sample_include_work_ids = setup.openalex_sample_include_work_ids

step_1_data_dir = project_folder / "step_1_data"
step_1_artifacts_dir = project_folder / "step_1_artifacts"
raw_dir = step_1_data_dir / "raw"
intermediate_dir = step_1_data_dir / "intermediate"
prepared_dir = step_1_data_dir / "prepared"
summary_tables_dir = step_1_artifacts_dir / "summary_tables"
dependency_tables_dir = step_1_artifacts_dir / "dependency_tables"
check_tables_dir = step_1_artifacts_dir / "check_tables"

raw_dir.mkdir(parents=True, exist_ok=True)
intermediate_dir.mkdir(parents=True, exist_ok=True)
prepared_dir.mkdir(parents=True, exist_ok=True)
summary_tables_dir.mkdir(parents=True, exist_ok=True)
dependency_tables_dir.mkdir(parents=True, exist_ok=True)
check_tables_dir.mkdir(parents=True, exist_ok=True)

TODAY = date.today().isoformat()
RUN_FROM_CACHE = use_existing_data

print(project_folder)
print(f"Run mode: {run_mode}")


/Users/endersari/2026-02-citations-vs-pc-memberships
Run mode: fast


## 2 - HotCRP URLs

In [3]:
urls_hotcrp = {
    "ICFP": {
        2017: "https://icfp17.hotcrp.com/users/pc",
        2018: "https://icfp18.hotcrp.com/users/pc",
        2019: "https://icfp19.hotcrp.com/users/pc",
        2020: "https://icfp20.hotcrp.com/users/pc",
        2021: "https://icfp21.hotcrp.com/users/pc",
        2022: "https://icfp22.hotcrp.com/users/pc",
        2023: "https://icfp23.hotcrp.com/users/pc",
        2024: "https://icfp24.hotcrp.com/users/pc",
        2025: "https://icfp25.hotcrp.com/users/pc",
    },
    "POPL": {
        2017: "https://popl17.hotcrp.com/users/pc",
        2018: "https://popl18.hotcrp.com/users/pc",
        2019: "https://popl19.hotcrp.com/users/pc",
        2020: "https://popl20.hotcrp.com/users/pc",
        2021: "https://popl21.hotcrp.com/users/pc",
        2022: "https://popl22.hotcrp.com/users/pc",
        2023: "https://popl23.hotcrp.com/users/pc",
        2024: "https://popl24.hotcrp.com/users/pc",
        2025: "https://popl25.hotcrp.com/users/pc",
    },
    "OOPSLA": {
        2017: "https://oopsla17.hotcrp.com/users/pc",
        2018: "https://oopsla18.hotcrp.com/users/pc",
        2019: "https://oopsla19.hotcrp.com/users/pc",
        2020: "https://oopsla20.hotcrp.com/users/pc",
        2021: "https://oopsla21.hotcrp.com/users/pc",
        2022: "https://oopsla22.hotcrp.com/users/pc",
        2023: "https://oopsla23.hotcrp.com/users/pc",
        2024: "https://oopsla24.hotcrp.com/users/pc",
        2025: "https://oopsla2425.hotcrp.com/users/pc",
    },
    "PLDI": {
        2017: "https://pldi17.hotcrp.com/users/pc",
        2018: "https://pldi18.hotcrp.com/users/pc",
        2019: "https://pldi19.hotcrp.com/users/pc",
        2020: "https://pldi20.hotcrp.com/users/pc",
        2021: "https://pldi21.hotcrp.com/users/pc",
        2022: "https://pldi22.hotcrp.com/users/pc",
        2023: "https://pldi23.hotcrp.com/users/pc",
        2024: "https://pldi24.hotcrp.com/users/pc",
        2025: "https://pldi25.hotcrp.com/users/pc",
    },
}


In [4]:
url_rows = []

for conf, year_to_url in urls_hotcrp.items():
    for year, url in year_to_url.items():
        url_rows.append({
            "conference": conf,
            "year": year,
            "url": url,
        })

url_df = pd.DataFrame(url_rows).sort_values(["conference", "year"])
print(url_df.shape)
display(url_df)


(36, 3)


,conference,year,url
0,ICFP,2017,https://icfp17.hotcrp.com/users/pc
1,ICFP,2018,https://icfp18.hotcrp.com/users/pc
2,ICFP,2019,https://icfp19.hotcrp.com/users/pc
3,ICFP,2020,https://icfp20.hotcrp.com/users/pc
4,ICFP,2021,https://icfp21.hotcrp.com/users/pc
5,ICFP,2022,https://icfp22.hotcrp.com/users/pc
6,ICFP,2023,https://icfp23.hotcrp.com/users/pc
7,ICFP,2024,https://icfp24.hotcrp.com/users/pc
8,ICFP,2025,https://icfp25.hotcrp.com/users/pc
18,OOPSLA,2017,https://oopsla17.hotcrp.com/users/pc


## 3 - Parser

In [5]:
def parse_hotcrp_pc(html):
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")

    rows = []
    if table is None:
        return pd.DataFrame(columns=["name", "role", "affiliation"])

    for tr in table.find_all("tr"):
        cells = tr.find_all(["th", "td"])

        if len(cells) < 2:
            continue
        if cells[0].name == "th":
            continue

        name_cell = cells[0]
        role_tag = name_cell.find(class_="pcrole")

        if role_tag is None:
            role = "PC Member"
        else:
            role_text = role_tag.get_text(strip=True).lower()
            if "associate" in role_text and "chair" in role_text:
                role = "Associate Chair"
            elif "chair" in role_text:
                role = "PC Chair"
            elif "sysadmin" in role_text:
                role = "Sysadmin"
            else:
                role = role_text.title()

        name = name_cell.get_text(" ", strip=True)
        if role_tag is not None:
            name = name.replace(role_tag.get_text(strip=True), "").strip()
        name = re.sub(r"\s+", " ", name)

        affiliation = cells[1].get_text(" ", strip=True)
        affiliation = re.sub(r"\s+", " ", affiliation)

        rows.append({
            "name": name,
            "role": role,
            "affiliation": affiliation,
        })

    return pd.DataFrame(rows)


## 4 - Fetch Pages

In [6]:
raw_dir

PosixPath('/Users/endersari/2026-02-citations-vs-pc-memberships/step_1_data/raw')

In [7]:
def cache_path(conf, year):
    folder = raw_dir / conf.lower() / "hotcrp_htmls"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / f"{conf.lower()}{year}_hotcrp_{TODAY}.html"


def latest_cached_page(conf, year):
    folder = raw_dir / conf.lower() / "hotcrp_htmls"
    files = sorted(folder.glob(f"{conf.lower()}{year}_hotcrp_*.html"))
    if len(files) == 0:
        return None
    return files[-1]


def get_html(conf, year, url):
    if RUN_FROM_CACHE:
        cached = latest_cached_page(conf, year)
        if cached is not None:
            html = cached.read_text(encoding="utf-8", errors="replace")
            return html, "cache", str(cached), url

    response = requests.get(
        url,
        timeout=20,
        headers={"User-Agent": "Mozilla/5.0"},
    )
    response.raise_for_status()

    path = cache_path(conf, year)
    path.write_text(response.text, encoding="utf-8")

    return response.text, "fetched", str(path), response.url


In [8]:
all_members = []
summary_rows = []

for row in url_df.itertuples(index=False):
    conf = row.conference
    year = int(row.year)
    url = row.url

    print(f"{conf} {year}...", end=" ")

    try:
        html, status, path, final_url = get_html(conf, year, url)
        df_year = parse_hotcrp_pc(html)
        error = ""
        print(f"{status}, {len(df_year)} rows")
    except Exception as e:
        df_year = pd.DataFrame(columns=["name", "role", "affiliation"])
        status = "error"
        path = ""
        final_url = url
        error = repr(e)
        print("error")

    if len(df_year) > 0:
        df_year["conference"] = conf
        df_year["year"] = year
        df_year["source"] = "HotCRP"
        df_year["source_url"] = url
        df_year["final_url"] = final_url
        all_members.append(df_year)

    summary_rows.append({
        "conference": conf,
        "year": year,
        "url": url,
        "final_url": final_url,
        "status": status,
        "cache_path": path,
        "n_members": len(df_year),
        "error": error,
    })

hotcrp_members = pd.concat(all_members, ignore_index=True)
hotcrp_summary = pd.DataFrame(summary_rows)


ICFP 2017... cache, 23 rows
ICFP 2018... cache, 63 rows
ICFP 2019... cache, 72 rows
ICFP 2020... cache, 60 rows
ICFP 2021... cache, 30 rows
ICFP 2022... cache, 42 rows
ICFP 2023... cache, 55 rows
ICFP 2024... cache, 51 rows
ICFP 2025... cache, 64 rows
OOPSLA 2017... fetched, 59 rows
OOPSLA 2018... fetched, 56 rows
OOPSLA 2019... fetched, 66 rows
OOPSLA 2020... fetched, 93 rows
OOPSLA 2021... fetched, 108 rows
OOPSLA 2022... fetched, 89 rows
OOPSLA 2023... fetched, 144 rows
OOPSLA 2024... fetched, 148 rows
OOPSLA 2025... fetched, 114 rows
PLDI 2017... fetched, 106 rows
PLDI 2018... fetched, 112 rows
PLDI 2019... fetched, 111 rows
PLDI 2020... fetched, 115 rows
PLDI 2021... fetched, 127 rows
PLDI 2022... fetched, 120 rows
PLDI 2023... fetched, 114 rows
PLDI 2024... fetched, 139 rows
PLDI 2025... fetched, 137 rows
POPL 2017... fetched, 90 rows
POPL 2018... fetched, 52 rows
POPL 2019... fetched, 52 rows
POPL 2020... fetched, 53 rows
POPL 2021... fetched, 53 rows
POPL 2022... fetched, 56 ro

## 5 - Save Data

In [9]:
hotcrp_members = hotcrp_members[
    ["conference", "year", "source", "name", "role", "affiliation", "source_url", "final_url"]
].sort_values(["conference", "year", "name"]).reset_index(drop=True)

hotcrp_summary = hotcrp_summary.sort_values(["conference", "year"]).reset_index(drop=True)

hotcrp_members.to_parquet(intermediate_dir / "hotcrp_members.parquet", index=False)
hotcrp_summary.to_csv(dependency_tables_dir / "hotcrp_fetch_summary.csv", index=False)

print(hotcrp_members.shape)
display(hotcrp_members.head())


(3032, 8)


,conference,year,source,name,role,affiliation,source_url,final_url
0,ICFP,2017,HotCRP,Adam Chlipala,PC Member,MIT CSAIL,https://icfp17.hotcrp.com/users/pc,https://icfp17.hotcrp.com/users/pc
1,ICFP,2017,HotCRP,Alan Jeffrey,PC Member,Mozilla Research,https://icfp17.hotcrp.com/users/pc,https://icfp17.hotcrp.com/users/pc
2,ICFP,2017,HotCRP,Alexandra Silva,PC Member,University College London,https://icfp17.hotcrp.com/users/pc,https://icfp17.hotcrp.com/users/pc
3,ICFP,2017,HotCRP,Ben Lippmeier,PC Member,Digital Asset,https://icfp17.hotcrp.com/users/pc,https://icfp17.hotcrp.com/users/pc
4,ICFP,2017,HotCRP,Beta Ziliani,PC Member,"CONICET and FAMAF, Universidad Nacional de Cór...",https://icfp17.hotcrp.com/users/pc,https://icfp17.hotcrp.com/users/pc


## 6 - Quick Checks

In [10]:
count_table = hotcrp_summary.pivot_table(
    index="year",
    columns="conference",
    values="n_members",
    aggfunc="sum",
).astype(int)

display(count_table)


conference,ICFP,OOPSLA,PLDI,POPL
year,,,,
2017,23,59,106,90
2018,63,56,112,52
2019,72,66,111,52
2020,60,93,115,53
2021,30,108,127,53
2022,42,89,120,56
2023,55,144,114,84
2024,51,148,139,91
2025,64,114,137,83


In [11]:
print("Rows by conference")
print(hotcrp_members.groupby("conference").size())

print("\nUnique names by conference")
print(hotcrp_members.groupby("conference")["name"].nunique())

print("\nFetch status")
print(hotcrp_summary["status"].value_counts())


Rows by conference
conference
ICFP       460
OOPSLA     877
PLDI      1081
POPL       614
dtype: int64

Unique names by conference
conference
ICFP      316
OOPSLA    540
PLDI      575
POPL      416
Name: name, dtype: int64

Fetch status
status
fetched    27
cache       9
Name: count, dtype: int64


## 7 - What I learned

HotCRP gives me the review system list. This is useful for validation and for
understanding review participation.

But this is not automatically the same as the PC list visible on the website. In the
next step, I compare these HotCRP names with the conference website lists.
